In [6]:
# ============================================================
# GenD TEST-ONLY evaluation: WildDeepfake -> WildDeepfake
# UPDATED VERSION WITH PROGRESS BAR + FIRST-BATCH CHECK
#
# Computes:
#   Frame metrics
#   Video metrics
#   EER
#   Uncertain Rate
#   Inference time
#
# Does NOT retrain.
# ============================================================

from pathlib import Path
import sys
import time
import inspect
import importlib.util
import numpy as np
import pandas as pd
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_fscore_support,
    roc_curve,
)

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


# ------------------------------------------------------------
# Paths and settings
# ------------------------------------------------------------

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")
GEND_SCRIPT = PROJECT_ROOT / "gend_reimpl_ffpp.py"

GEND_OUT = PROJECT_ROOT / "outputs_gend_wilddeepfake_train"

CKPT_CANDIDATES = [
    GEND_OUT / "best_checkpoint.pt",
    GEND_OUT / "latest_checkpoint.pt",
    GEND_OUT / "gend_best.pt",
    GEND_OUT / "gend_latest.pt",
    GEND_OUT / "checkpoint_best.pt",
    GEND_OUT / "checkpoint_latest.pt",
]

CKPT_PATH = None
for p in CKPT_CANDIDATES:
    if p.exists():
        CKPT_PATH = p
        break

if CKPT_PATH is None:
    raise FileNotFoundError(
        "No GenD WildDeepfake checkpoint found. Checked:\n" +
        "\n".join(str(p) for p in CKPT_CANDIDATES)
    )

TEST_MANIFEST = PROJECT_ROOT / "outputs_wilddeepfake/wilddeepfake_manifest_canonical_gend_safe.csv"
if not TEST_MANIFEST.exists():
    TEST_MANIFEST = PROJECT_ROOT / "outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv"

OUT_DIR = PROJECT_ROOT / "outputs_gend_wilddeepfake_indomain_test_eval_progress"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT = "test"
IMAGE_SIZE = 224
FRAMES_PER_VIDEO = 32
BATCH_SIZE = 16
NUM_WORKERS = 0
UNCERTAINTY_CONFIDENCE_THRESHOLD = 0.60
SEED = 42

# Set to a number such as 50 for a quick debug run.
# Keep None for the full in-domain test.
MAX_VIDEOS = None

PARTIAL_SAVE_EVERY_N_BATCHES = 100

print("=" * 100)
print("GenD WildDeepfake -> WildDeepfake in-domain test evaluation")
print("=" * 100)
print("GenD script:", GEND_SCRIPT)
print("Checkpoint:", CKPT_PATH)
print("Test manifest:", TEST_MANIFEST)
print("Output dir:", OUT_DIR)
print("Frames per video:", FRAMES_PER_VIDEO)
print("Batch size:", BATCH_SIZE)


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def normalize_manifest(path, split="test"):
    df = pd.read_csv(path)

    if "frame_path" not in df.columns:
        if "image_path" in df.columns:
            df["frame_path"] = df["image_path"].astype(str)
        else:
            raise ValueError("Manifest needs frame_path or image_path.")

    if "video_id" not in df.columns:
        if "sample_unit_id" in df.columns:
            df["video_id"] = df["sample_unit_id"].astype(str)
        elif "group_id" in df.columns:
            df["video_id"] = df["group_id"].astype(str)
        else:
            df["video_id"] = df["frame_path"].apply(lambda x: str(Path(x).parent))

    required = ["frame_path", "split", "label", "video_id"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing manifest columns: {missing}")

    df["frame_path"] = df["frame_path"].astype(str)
    df["video_id"] = df["video_id"].astype(str)
    df["split"] = df["split"].astype(str)
    df["label"] = df["label"].astype(int)

    df = df[df["split"] == split].copy()

    if len(df) == 0:
        raise RuntimeError(f"No rows found for split={split}")

    return df


def sample_frames_per_video(df, frames_per_video=32):
    parts = []

    for vid, g in df.groupby("video_id", sort=False):
        g = g.sort_values("frame_path").reset_index(drop=True)

        if len(g) <= frames_per_video:
            parts.append(g)
        else:
            idx = np.linspace(0, len(g) - 1, frames_per_video).round().astype(int)
            parts.append(g.iloc[idx].copy())

    out = pd.concat(parts, ignore_index=True)
    out = out.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    return out


class ManifestImageDataset(Dataset):
    def __init__(self, df, image_size=224):
        self.df = df.reset_index(drop=True)
        self.image_size = image_size
        self.tf = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row["frame_path"]
        label = int(row["label"])
        video_id = str(row["video_id"])

        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            img = Image.new("RGB", (self.image_size, self.image_size))

        return {
            "image": self.tf(img),
            "label": torch.tensor(label, dtype=torch.long),
            "path": path,
            "video_id": video_id,
        }


def import_gend_module(script_path):
    spec = importlib.util.spec_from_file_location("gend_reimpl_ffpp_runtime", str(script_path))
    module = importlib.util.module_from_spec(spec)
    sys.modules["gend_reimpl_ffpp_runtime"] = module
    spec.loader.exec_module(module)
    return module


def get_state_dict_from_checkpoint(ckpt):
    if isinstance(ckpt, dict):
        for key in ["model", "model_state_dict", "state_dict", "net", "network"]:
            if key in ckpt and isinstance(ckpt[key], dict):
                print("Using checkpoint state key:", key)
                return ckpt[key]

    if isinstance(ckpt, dict):
        print("Using checkpoint directly as state_dict.")
        return ckpt

    raise RuntimeError(f"Unsupported checkpoint type: {type(ckpt)}")


def clean_state_dict_keys(state):
    cleaned = {}
    for k, v in state.items():
        nk = str(k)
        for prefix in ["module.", "model.", "net.", "network."]:
            if nk.startswith(prefix):
                nk = nk[len(prefix):]
        cleaned[nk] = v
    return cleaned


def build_gend_model(gend_module, ckpt):
    if not hasattr(gend_module, "GenDReimpl"):
        raise AttributeError("Could not find GenDReimpl in gend_reimpl_ffpp.py")

    ModelCls = gend_module.GenDReimpl

    cfg = {}
    if isinstance(ckpt, dict):
        for key in ["config", "args", "hparams", "hyperparameters"]:
            if key in ckpt and isinstance(ckpt[key], dict):
                cfg.update(ckpt[key])

    sig = inspect.signature(ModelCls)
    kwargs = {}

    for name in sig.parameters:
        if name == "self":
            continue
        if name in cfg:
            kwargs[name] = cfg[name]
        elif name == "backbone":
            kwargs[name] = cfg.get("backbone", "resnet18")
        elif name == "pretrained":
            kwargs[name] = False
        elif name == "num_classes":
            kwargs[name] = 2
        elif name == "train_ln":
            kwargs[name] = cfg.get("train_ln", False)
        elif name == "dropout":
            kwargs[name] = cfg.get("dropout", 0.0)

    print("GenD constructor kwargs:", kwargs)

    try:
        model = ModelCls(**kwargs)
    except Exception as e:
        print("Constructor with kwargs failed:", repr(e))
        print("Trying GenDReimpl() with defaults.")
        model = ModelCls()

    return model


def extract_logits(output):
    if torch.is_tensor(output):
        return output

    if isinstance(output, dict):
        for key in [
            "logits", "cls", "pred", "prediction", "output", "out",
            "y_pred", "final_output", "prob"
        ]:
            if key in output and torch.is_tensor(output[key]):
                return output[key]

        for value in output.values():
            if torch.is_tensor(value) and value.ndim >= 1:
                return value

    if isinstance(output, (list, tuple)):
        for value in output:
            if torch.is_tensor(value):
                return value

        for value in output:
            try:
                return extract_logits(value)
            except Exception:
                pass

    raise RuntimeError(f"Could not extract logits from output type: {type(output)}")


def forward_gend(model, images):
    attempts = [
        ("model(images)", lambda: model(images)),
        ("model({'image': images})", lambda: model({"image": images})),
        ("model({'img': images})", lambda: model({"img": images})),
        ("model({'x': images})", lambda: model({"x": images})),
    ]

    errors = []
    for name, fn in attempts:
        try:
            output = fn()
            logits = extract_logits(output)

            if logits.ndim == 1:
                logits = logits.view(-1, 1)

            if logits.ndim > 2:
                logits = logits.view(logits.shape[0], -1)

            return logits

        except Exception as e:
            errors.append(f"{name}: {repr(e)}")

    raise RuntimeError("GenD forward failed. Errors:\n" + "\n".join(errors))


def fake_scores_from_logits(logits):
    if logits.shape[1] == 1:
        return torch.sigmoid(logits[:, 0])
    return torch.softmax(logits[:, :2], dim=1)[:, 1]


def compute_eer(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)

    if len(np.unique(y_true)) < 2:
        return np.nan

    fpr, tpr, _ = roc_curve(y_true, y_score)
    fnr = 1.0 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    return float((fpr[idx] + fnr[idx]) / 2.0)


def compute_uncertain_rate(y_score, threshold=0.60):
    y_score = np.asarray(y_score).astype(float)
    confidence = np.maximum(y_score, 1.0 - y_score)
    return float(np.mean(confidence < threshold) * 100.0)


def compute_metric_row(y_true, y_score, level, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    y_pred = (y_score >= threshold).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1],
        zero_division=0,
    )

    return {
        "Level": level,
        "Decision Threshold": threshold,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "AUPRC": average_precision_score(y_true, y_score) if len(np.unique(y_true)) == 2 else np.nan,
        "AUCROC": roc_auc_score(y_true, y_score) if len(np.unique(y_true)) == 2 else np.nan,
        "EER (Equal Error Rate)": compute_eer(y_true, y_score),
        "Precision (0)": precision[0],
        "Recall (0)": recall[0],
        "F1 (0)": f1[0],
        "Precision (1)": precision[1],
        "Recall (1)": recall[1],
        "F1 (1)": f1[1],
        "Uncertain Rate (%)": compute_uncertain_rate(
            y_score,
            threshold=UNCERTAINTY_CONFIDENCE_THRESHOLD,
        ),
        "Predicted Real Count": int((y_pred == 0).sum()),
        "Predicted Fake Count": int((y_pred == 1).sum()),
        "Rows": int(len(y_true)),
    }


# ------------------------------------------------------------
# Load WildDeepfake test split
# ------------------------------------------------------------

test_df = normalize_manifest(TEST_MANIFEST, split=SPLIT)

if MAX_VIDEOS is not None:
    keep_videos = test_df["video_id"].drop_duplicates().head(MAX_VIDEOS)
    test_df = test_df[test_df["video_id"].isin(keep_videos)].copy()
    print(f"\nDEBUG MODE: limited to {MAX_VIDEOS} videos before frame sampling.")

test_df = sample_frames_per_video(test_df, frames_per_video=FRAMES_PER_VIDEO)

test_df.to_csv(OUT_DIR / "test_manifest_used.csv", index=False)

print("\nWildDeepfake test rows after sampling:", len(test_df))
print("WildDeepfake test videos:", test_df["video_id"].nunique())
print("\nTest counts:")
display(pd.crosstab(test_df["split"], test_df["label"]))


# ------------------------------------------------------------
# Load GenD model
# ------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nDevice:", device)

gend_module = import_gend_module(GEND_SCRIPT)
ckpt = torch.load(CKPT_PATH, map_location=device)

if isinstance(ckpt, dict):
    print("Top-level checkpoint keys:", list(ckpt.keys())[:20])

model = build_gend_model(gend_module, ckpt).to(device)

state = clean_state_dict_keys(get_state_dict_from_checkpoint(ckpt))
missing, unexpected = model.load_state_dict(state, strict=False)

print("\nLoaded GenD checkpoint.")
print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

if len(missing) > 0:
    print("First missing keys:", missing[:10])
if len(unexpected) > 0:
    print("First unexpected keys:", unexpected[:10])

vision_missing = [k for k in missing if str(k).startswith("vision.")]
nonvision_missing = [k for k in missing if not str(k).startswith("vision.")]

print("Missing vision.* keys:", len(vision_missing))
print("Missing non-vision keys:", len(nonvision_missing))

if len(nonvision_missing) > 0:
    print("First non-vision missing keys:", nonvision_missing[:20])

model.eval()


# ------------------------------------------------------------
# Build DataLoader
# ------------------------------------------------------------

dataset = ManifestImageDataset(test_df, image_size=IMAGE_SIZE)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

print("\nEvaluation batches:", len(loader))
print("Evaluation images:", len(dataset))
print("Batch size:", BATCH_SIZE)
print("Output dir:", OUT_DIR)


# ------------------------------------------------------------
# First-batch smoke test
# ------------------------------------------------------------

print("\nRunning first-batch forward check...")

first_batch = next(iter(loader))
first_images = first_batch["image"].to(device)

with torch.inference_mode():
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    first_logits = forward_gend(model, first_images)
    first_scores = fake_scores_from_logits(first_logits)
    if device.type == "cuda":
        torch.cuda.synchronize()
    first_elapsed = time.time() - t0

print("First batch logits shape:", tuple(first_logits.shape))
print("First batch scores shape:", tuple(first_scores.shape))
print("First batch elapsed sec:", round(first_elapsed, 4))
print("First batch p_fake range:", float(first_scores.min()), "to", float(first_scores.max()))

if device.type == "cuda":
    print("\nGPU memory after first batch:")
    print("Allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 3))
    print("Reserved GB:", round(torch.cuda.memory_reserved() / 1024**3, 3))


# ------------------------------------------------------------
# Evaluate full test split
# ------------------------------------------------------------

all_rows = []
y_true = []
y_score = []

start_total = time.time()
num_images = 0

iterator = loader
if tqdm is not None:
    iterator = tqdm(loader, desc="Evaluating GenD WildDeepfake test", total=len(loader))

with torch.inference_mode():
    for batch_idx, batch in enumerate(iterator, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        if device.type == "cuda":
            torch.cuda.synchronize()

        t0 = time.time()
        logits = forward_gend(model, images)

        if device.type == "cuda":
            torch.cuda.synchronize()

        elapsed = time.time() - t0

        scores = fake_scores_from_logits(logits).detach().cpu().numpy()
        labels_np = labels.detach().cpu().numpy()

        paths = batch["path"]
        vids = batch["video_id"]

        bs = len(labels_np)
        num_images += bs

        for i in range(bs):
            p_fake = float(scores[i])
            confidence = max(p_fake, 1.0 - p_fake)
            pred = int(p_fake >= 0.5)
            uncertain = int(confidence < UNCERTAINTY_CONFIDENCE_THRESHOLD)

            all_rows.append({
                "image_path": paths[i],
                "video_id": vids[i],
                "label": int(labels_np[i]),
                "p_fake": p_fake,
                "prediction": pred,
                "confidence": confidence,
                "uncertain": uncertain,
                "inference_time_sec_per_image_batch_avg": elapsed / max(bs, 1),
            })

        y_true.extend(labels_np.tolist())
        y_score.extend(scores.tolist())

        if batch_idx % PARTIAL_SAVE_EVERY_N_BATCHES == 0:
            partial_path = OUT_DIR / "test_frame_predictions_partial.csv"
            pd.DataFrame(all_rows).to_csv(partial_path, index=False)

            elapsed_total = time.time() - start_total
            ips = num_images / max(elapsed_total, 1e-9)

            print(
                f"\nProgress batch {batch_idx}/{len(loader)} | "
                f"images={num_images}/{len(dataset)} | "
                f"images/sec={ips:.3f} | "
                f"partial saved={partial_path}"
            )

total_elapsed = time.time() - start_total

print("\nFinished GenD evaluation.")
print("Total images:", num_images)
print("Total elapsed sec:", round(total_elapsed, 2))
print("Images/sec:", round(num_images / max(total_elapsed, 1e-9), 4))


# ------------------------------------------------------------
# Save frame predictions
# ------------------------------------------------------------

frame_pred_df = pd.DataFrame(all_rows)
frame_pred_df.to_csv(OUT_DIR / "test_frame_predictions.csv", index=False)


# ------------------------------------------------------------
# Video-level aggregation
# ------------------------------------------------------------

video_pred_df = (
    frame_pred_df
    .groupby("video_id", as_index=False)
    .agg(
        label=("label", "first"),
        p_fake=("p_fake", "mean"),
        frame_count=("p_fake", "size"),
    )
)

video_pred_df["prediction"] = (video_pred_df["p_fake"] >= 0.5).astype(int)
video_pred_df["confidence"] = np.maximum(video_pred_df["p_fake"], 1.0 - video_pred_df["p_fake"])
video_pred_df["uncertain"] = (
    video_pred_df["confidence"] < UNCERTAINTY_CONFIDENCE_THRESHOLD
).astype(int)

video_pred_df.to_csv(OUT_DIR / "test_video_predictions.csv", index=False)


# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

frame_metrics = compute_metric_row(
    frame_pred_df["label"].values,
    frame_pred_df["p_fake"].values,
    level="Frame",
)

video_metrics = compute_metric_row(
    video_pred_df["label"].values,
    video_pred_df["p_fake"].values,
    level="Video",
)

metrics_df = pd.DataFrame([frame_metrics, video_metrics])
metrics_df.to_csv(OUT_DIR / "test_metrics.csv", index=False)

time_df = pd.DataFrame([{
    "num_images": int(num_images),
    "total_inference_time_sec": total_elapsed,
    "avg_inference_time_sec_per_image": total_elapsed / max(num_images, 1),
    "images_per_second": num_images / max(total_elapsed, 1e-9),
    "batch_size": BATCH_SIZE,
    "image_size": IMAGE_SIZE,
    "frames_per_video": FRAMES_PER_VIDEO,
    "num_workers": NUM_WORKERS,
    "checkpoint": str(CKPT_PATH),
    "test_manifest": str(TEST_MANIFEST),
    "output_dir": str(OUT_DIR),
}])
time_df.to_csv(OUT_DIR / "test_inference_time.csv", index=False)


# ------------------------------------------------------------
# Print final results
# ------------------------------------------------------------

print("\nSaved:")
print(OUT_DIR / "test_metrics.csv")
print(OUT_DIR / "test_frame_predictions.csv")
print(OUT_DIR / "test_video_predictions.csv")
print(OUT_DIR / "test_inference_time.csv")

print("\nGenD in-domain WildDeepfake test metrics:")
display(metrics_df)

print("\nInference time:")
display(time_df)

GenD WildDeepfake -> WildDeepfake in-domain test evaluation
GenD script: /home/jovyan/Deepfake_Low Resource_Low Resolution/gend_reimpl_ffpp.py
Checkpoint: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_gend_wilddeepfake_train/best_checkpoint.pt
Test manifest: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_manifest_canonical_gend_safe.csv
Output dir: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_gend_wilddeepfake_indomain_test_eval_progress
Frames per video: 32
Batch size: 16

WildDeepfake test rows after sampling: 25792
WildDeepfake test videos: 806

Test counts:


label,0,1
split,,
test,12672,13120



Device: cuda
Top-level checkpoint keys: ['epoch', 'threshold', 'best_metric', 'args', 'state_dict']
GenD constructor kwargs: {'backbone': 'openai/clip-vit-large-patch14', 'train_ln': False}


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 22649.68it/s]
[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight  

Using checkpoint state key: state_dict

Loaded GenD checkpoint.
Missing keys: 291
Unexpected keys: 0
First missing keys: ['vision.embeddings.class_embedding', 'vision.embeddings.patch_embedding.weight', 'vision.embeddings.position_embedding.weight', 'vision.encoder.layers.0.self_attn.k_proj.weight', 'vision.encoder.layers.0.self_attn.k_proj.bias', 'vision.encoder.layers.0.self_attn.v_proj.weight', 'vision.encoder.layers.0.self_attn.v_proj.bias', 'vision.encoder.layers.0.self_attn.q_proj.weight', 'vision.encoder.layers.0.self_attn.q_proj.bias', 'vision.encoder.layers.0.self_attn.out_proj.weight']
Missing vision.* keys: 291
Missing non-vision keys: 0

Evaluation batches: 1612
Evaluation images: 25792
Batch size: 16
Output dir: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_gend_wilddeepfake_indomain_test_eval_progress

Running first-batch forward check...
First batch logits shape: (16, 2)
First batch scores shape: (16,)
First batch elapsed sec: 0.1386
First batch p_fake range:

Evaluating GenD WildDeepfake test:   6%|▌         | 100/1612 [01:41<22:53,  1.10it/s]

... (32 outputs left out)


Finished GenD evaluation.
Total images: 25792
Total elapsed sec: 2107.1
Images/sec: 12.2405

Saved:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_gend_wilddeepfake_indomain_test_eval_progress/test_metrics.csv
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_gend_wilddeepfake_indomain_test_eval_progress/test_frame_predictions.csv
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_gend_wilddeepfake_indomain_test_eval_progress/test_video_predictions.csv
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_gend_wilddeepfake_indomain_test_eval_progress/test_inference_time.csv

GenD in-domain WildDeepfake test metrics:


,Level,Decision Threshold,Accuracy,Balanced Accuracy,AUPRC,AUCROC,EER (Equal Error Rate),Precision (0),Recall (0),F1 (0),Precision (1),Recall (1),F1 (1),Uncertain Rate (%),Predicted Real Count,Predicted Fake Count,Rows
0,Frame,0.5,0.835996,0.836484,0.905168,0.905504,0.161873,0.813363,0.864583,0.838191,0.860737,0.808384,0.833739,4.478133,13470,12322,25792
1,Video,0.5,0.863524,0.864086,0.938373,0.934559,0.133992,0.837264,0.896465,0.865854,0.892670,0.831707,0.861111,7.320099,424,382,806



Inference time:


,num_images,total_inference_time_sec,avg_inference_time_sec_per_image,images_per_second,batch_size,image_size,frames_per_video,num_workers,checkpoint,test_manifest,output_dir
0,25792,2107.09506,0.081696,12.240549,16,224,32,0,/home/jovyan/Deepfake_Low Resource_Low Resolut...,/home/jovyan/Deepfake_Low Resource_Low Resolut...,/home/jovyan/Deepfake_Low Resource_Low Resolut...


In [ ]:
# ============================================================
# DFD-HR resume to epoch 5 using environment variable
#
# This avoids argparse patching completely.
# It resumes from latest_checkpoint.pt and stops after epoch 5.
# ============================================================

from pathlib import Path
import subprocess
import sys
import os
import shlex
import re
import shutil
import time
import torch
import pandas as pd

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")
DFDHR_ROOT = PROJECT_ROOT / "DFD-HR"
SCRIPT = PROJECT_ROOT / "train_dfdhr_manifest_wilddeepfake.py"
WILD_MANIFEST = PROJECT_ROOT / "outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv"

OUT_DIR = PROJECT_ROOT / "outputs_dfdhr_wilddeepfake_train_no_amp_cc"

LOG_PATH = OUT_DIR / "train_dfdhr_wilddeepfake_resume_to_epoch5.log"
PID_PATH = OUT_DIR / "train_dfdhr_wilddeepfake_resume_to_epoch5.pid"

TOTAL_EPOCHS = 5

# Keep these the same as your interrupted run.
BATCH_SIZE = 4
IMAGE_SIZE = 448
FRAMES_PER_VIDEO = 32
NUM_WORKERS = 0

OUT_DIR.mkdir(parents=True, exist_ok=True)

if not SCRIPT.exists():
    raise FileNotFoundError(SCRIPT)

if not WILD_MANIFEST.exists():
    raise FileNotFoundError(WILD_MANIFEST)

# ------------------------------------------------------------
# Pick checkpoint
# ------------------------------------------------------------

CKPT_CANDIDATES = [
    OUT_DIR / "latest_checkpoint.pt",
    OUT_DIR / "best_checkpoint.pt",
]

CKPT_PATH = None
ckpt_cpu = None

for p in CKPT_CANDIDATES:
    if not p.exists():
        continue
    try:
        ckpt_cpu = torch.load(p, map_location="cpu", weights_only=False)
        CKPT_PATH = p
        break
    except Exception as e:
        print("Could not load:", p)
        print(repr(e))

if CKPT_PATH is None:
    raise FileNotFoundError("No readable checkpoint found.")

print("Resume checkpoint:", CKPT_PATH)

# ------------------------------------------------------------
# Detect completed epoch
# ------------------------------------------------------------

completed_epoch = 0

if isinstance(ckpt_cpu, dict):
    print("Checkpoint keys:", list(ckpt_cpu.keys())[:30])
    for key in ["epoch", "current_epoch", "completed_epoch"]:
        if key in ckpt_cpu:
            try:
                completed_epoch = int(ckpt_cpu[key])
                break
            except Exception:
                pass

history_path = OUT_DIR / "training_history.csv"

if history_path.exists():
    hist = pd.read_csv(history_path)
    print("\nExisting training history:")
    display(hist.tail(10))

    backup_history = OUT_DIR / f"training_history_before_resume_{int(time.time())}.csv"
    shutil.copy2(history_path, backup_history)
    print("Backed up training history to:", backup_history)

    if completed_epoch <= 0:
        for col in ["epoch", "Epoch"]:
            if col in hist.columns:
                vals = pd.to_numeric(hist[col], errors="coerce").dropna()
                if len(vals) > 0:
                    completed_epoch = int(vals.max())
                    break

print("\nDetected completed epoch:", completed_epoch)
print("Resume will start at epoch:", completed_epoch + 1)

if completed_epoch >= TOTAL_EPOCHS:
    raise RuntimeError(f"Already completed epoch {completed_epoch}; target is {TOTAL_EPOCHS}.")

# ------------------------------------------------------------
# Patch trainer with environment-based resume support
# ------------------------------------------------------------

text = SCRIPT.read_text(errors="replace")

backup_script = SCRIPT.with_suffix(f".env_resume_backup_{int(time.time())}.py")
shutil.copy2(SCRIPT, backup_script)
print("\nBacked up trainer script to:", backup_script)

# Ensure imports.
if "from pathlib import Path" not in text:
    text = "from pathlib import Path\n" + text

if "import pandas as pd" not in text:
    text = "import pandas as pd\n" + text

if "import os" not in text:
    text = "import os\n" + text

# Patch history loading.
if "[resume-env] Loaded existing training history rows" not in text:
    history_pattern = r"(^[ \t]*)history\s*=\s*\[\]\s*$"

    def history_repl(match):
        ind = match.group(1)
        return f'''{ind}history = []
{ind}_resume_from_env = os.environ.get("DFDHR_RESUME_FROM")
{ind}_history_csv_resume = Path(args.out_dir) / "training_history.csv"
{ind}if _resume_from_env and _history_csv_resume.exists():
{ind}    try:
{ind}        history = pd.read_csv(_history_csv_resume).to_dict("records")
{ind}        print(f"[resume-env] Loaded existing training history rows: {{len(history)}}")
{ind}    except Exception as e:
{ind}        print(f"[resume-env] Could not load existing training history: {{repr(e)}}")
{ind}        history = []'''

    text2, n_hist = re.subn(history_pattern, history_repl, text, count=1, flags=re.M)

    if n_hist == 1:
        text = text2
        print("Patched history loading.")
    else:
        print("Could not find plain history = [] block. Continuing; history backup exists.")
else:
    print("History loading already patched.")

# Patch resume block after GradScaler line.
if "[resume-env] Loading checkpoint from" not in text:
    scaler_pattern = r"(^[ \t]*scaler\s*=\s*.*GradScaler\(.*?\)\s*$)"

    def scaler_repl(match):
        line = match.group(1)
        ind = re.match(r"^([ \t]*)", line).group(1)

        block = f'''
{ind}start_epoch = 1
{ind}_resume_from_env = os.environ.get("DFDHR_RESUME_FROM")
{ind}if _resume_from_env:
{ind}    resume_path = Path(_resume_from_env)
{ind}    if not resume_path.exists():
{ind}        raise FileNotFoundError(f"Resume checkpoint not found: {{resume_path}}")
{ind}
{ind}    print(f"[resume-env] Loading checkpoint from: {{resume_path}}")
{ind}    resume_ckpt = torch.load(resume_path, map_location=device, weights_only=False)
{ind}
{ind}    resume_state = None
{ind}    if isinstance(resume_ckpt, dict):
{ind}        for resume_key in ["model", "model_state_dict", "state_dict", "net", "network"]:
{ind}            if resume_key in resume_ckpt and isinstance(resume_ckpt[resume_key], dict):
{ind}                resume_state = resume_ckpt[resume_key]
{ind}                print(f"[resume-env] Using checkpoint model key: {{resume_key}}")
{ind}                break
{ind}
{ind}    if resume_state is None:
{ind}        resume_state = resume_ckpt
{ind}
{ind}    missing, unexpected = model.load_state_dict(resume_state, strict=False)
{ind}    print(f"[resume-env] Loaded model state. Missing keys={{len(missing)}} Unexpected keys={{len(unexpected)}}")
{ind}
{ind}    if isinstance(resume_ckpt, dict) and "optimizer" in resume_ckpt:
{ind}        try:
{ind}            optimizer.load_state_dict(resume_ckpt["optimizer"])
{ind}            print("[resume-env] Loaded optimizer state.")
{ind}        except Exception as e:
{ind}            print(f"[resume-env] Could not load optimizer state; using fresh optimizer: {{repr(e)}}")
{ind}
{ind}    if isinstance(resume_ckpt, dict) and "scaler" in resume_ckpt:
{ind}        try:
{ind}            scaler.load_state_dict(resume_ckpt["scaler"])
{ind}            print("[resume-env] Loaded scaler state.")
{ind}        except Exception as e:
{ind}            print(f"[resume-env] Could not load scaler state; continuing: {{repr(e)}}")
{ind}
{ind}    completed_epoch = 0
{ind}    if isinstance(resume_ckpt, dict):
{ind}        for epoch_key in ["epoch", "current_epoch", "completed_epoch"]:
{ind}            if epoch_key in resume_ckpt:
{ind}                try:
{ind}                    completed_epoch = int(resume_ckpt[epoch_key])
{ind}                    break
{ind}                except Exception:
{ind}                    pass
{ind}
{ind}    if completed_epoch <= 0:
{ind}        hist_csv = Path(args.out_dir) / "training_history.csv"
{ind}        if hist_csv.exists():
{ind}            try:
{ind}                hist_df = pd.read_csv(hist_csv)
{ind}                for col in ["epoch", "Epoch"]:
{ind}                    if col in hist_df.columns:
{ind}                        vals = pd.to_numeric(hist_df[col], errors="coerce").dropna()
{ind}                        if len(vals) > 0:
{ind}                            completed_epoch = int(vals.max())
{ind}                            break
{ind}            except Exception as e:
{ind}                print(f"[resume-env] Could not infer epoch from history: {{repr(e)}}")
{ind}
{ind}    start_epoch = max(1, completed_epoch + 1)
{ind}    print(f"[resume-env] Completed epoch={{completed_epoch}}. Starting at epoch={{start_epoch}}. Target total epochs={{args.epochs}}.")
{ind}else:
{ind}    print("[resume-env] No DFDHR_RESUME_FROM set. Starting at epoch 1.")
'''
        return line + block

    text2, n_scaler = re.subn(scaler_pattern, scaler_repl, text, count=1, flags=re.M)

    if n_scaler != 1:
        print("\nCould not find GradScaler line. Lines containing scaler:")
        for i, line in enumerate(text.splitlines()):
            if "scaler" in line or "GradScaler" in line:
                print(f"{i+1:04d}: {line}")
        raise RuntimeError("Could not insert resume block.")

    text = text2
    print("Patched checkpoint resume block.")
else:
    print("Checkpoint resume block already present.")

# Patch training loop to use start_epoch.
if "range(start_epoch, args.epochs + 1)" not in text:
    text2, n_loop = re.subn(
        r"for epoch in range\(\s*1\s*,\s*args\.epochs\s*\+\s*1\s*\):",
        "for epoch in range(start_epoch, args.epochs + 1):",
        text,
        count=1,
    )

    if n_loop != 1:
        print("\nCould not find exact epoch loop. Lines containing for epoch:")
        for i, line in enumerate(text.splitlines()):
            if "for epoch" in line:
                print(f"{i+1:04d}: {line}")
        raise RuntimeError("Could not patch epoch loop.")

    text = text2
    print("Patched epoch loop to use start_epoch.")
else:
    print("Epoch loop already uses start_epoch.")

SCRIPT.write_text(text)
print("\nPatched trainer saved:", SCRIPT)

# ------------------------------------------------------------
# Compiler env for Triton
# ------------------------------------------------------------

compiler_check = subprocess.run(
    ["bash", "-lc", "which x86_64-conda-linux-gnu-gcc || which gcc || which cc || true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
).stdout.strip()

if not compiler_check:
    print("\nNo compiler found. Installing conda compilers...")
    subprocess.run(
        ["bash", "-lc", "conda install -y -c conda-forge gcc_linux-64 gxx_linux-64"],
        check=True,
    )

gcc_path = subprocess.run(
    ["bash", "-lc", "which x86_64-conda-linux-gnu-gcc || which gcc || which cc"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=True,
).stdout.strip().splitlines()[0]

gxx_result = subprocess.run(
    ["bash", "-lc", "which x86_64-conda-linux-gnu-g++ || which g++ || true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
).stdout.strip().splitlines()

gxx_path = gxx_result[0] if gxx_result else gcc_path

print("\nUsing CC :", gcc_path)
print("Using CXX:", gxx_path)

# ------------------------------------------------------------
# Launch resume run
# ------------------------------------------------------------

existing = subprocess.run(
    ["bash", "-lc", "ps aux | grep -i 'train_dfdhr_manifest_wilddeepfake.py' | grep -v grep || true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
).stdout.strip()

if existing:
    print("\nA DFD-HR process is already running. Not launching duplicate:")
    print(existing)
else:
    cmd = [
        sys.executable,
        "-u",
        str(SCRIPT),
        "--repo_root", str(DFDHR_ROOT),
        "--metadata_csv", str(WILD_MANIFEST),
        "--out_dir", str(OUT_DIR),
        "--epochs", str(TOTAL_EPOCHS),
        "--batch_size", str(BATCH_SIZE),
        "--image_size", str(IMAGE_SIZE),
        "--frames_per_video", str(FRAMES_PER_VIDEO),
        "--num_workers", str(NUM_WORKERS),
        # no --resume_from needed
        # no --amp
    ]

    cuda_home = os.environ.get("CUDA_HOME", "/usr/local/cuda")

    print("\nLaunching resumed DFD-HR training:")
    print(" ".join(shlex.quote(x) for x in cmd))
    print("Resume env DFDHR_RESUME_FROM:", CKPT_PATH)
    print("Target total epochs:", TOTAL_EPOCHS)
    print("Log:", LOG_PATH)

    bash_cmd = (
        "cd " + shlex.quote(str(PROJECT_ROOT)) + " && "
        "DFDHR_RESUME_FROM=" + shlex.quote(str(CKPT_PATH)) + " "
        "CC=" + shlex.quote(gcc_path) + " "
        "CXX=" + shlex.quote(gxx_path) + " "
        "CUDA_HOME=" + shlex.quote(cuda_home) + " "
        "nohup " + " ".join(shlex.quote(x) for x in cmd) +
        " > " + shlex.quote(str(LOG_PATH)) + " 2>&1 & echo $!"
    )

    proc = subprocess.run(
        ["bash", "-lc", bash_cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    print(proc.stdout)

    pid = proc.stdout.strip().splitlines()[-1]
    PID_PATH.write_text(pid)

    print("Launched DFD-HR resume PID:", pid)
    print("PID saved to:", PID_PATH)
    print("Log:", LOG_PATH)

Resume checkpoint: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wilddeepfake_train_no_amp_cc/latest_checkpoint.pt
Checkpoint keys: ['model', 'optimizer', 'epoch', 'config', 'args', 'history']

Existing training history:


,epoch,train_loss,train_acc,val_loss,val_Accuracy,val_Balanced Accuracy,val_AUPRC,val_AUCROC,val_Precision (0),val_Recall (0),val_F1 (0),val_Precision (1),val_Recall (1),val_F1 (1),elapsed_min
0,1,0.105614,0.960710,0.166057,0.957133,0.956557,0.984768,0.987476,0.950454,0.968658,0.959470,0.964782,0.944456,0.954511,1577.744576
1,2,0.050913,0.982132,0.182802,0.949501,0.949262,0.984005,0.985173,0.949571,0.954271,0.951915,0.949422,0.944254,0.946831,1577.577095


Backed up training history to: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wilddeepfake_train_no_amp_cc/training_history_before_resume_1788141456.csv

Detected completed epoch: 2
Resume will start at epoch: 3

Backed up trainer script to: /home/jovyan/Deepfake_Low Resource_Low Resolution/train_dfdhr_manifest_wilddeepfake.env_resume_backup_1788141456.py
History loading already patched.
Checkpoint resume block already present.
Epoch loop already uses start_epoch.

Patched trainer saved: /home/jovyan/Deepfake_Low Resource_Low Resolution/train_dfdhr_manifest_wilddeepfake.py

Using CC : /opt/conda/bin/x86_64-conda-linux-gnu-gcc
Using CXX: /opt/conda/bin/x86_64-conda-linux-gnu-g++

Launching resumed DFD-HR training:
/opt/conda/bin/python -u '/home/jovyan/Deepfake_Low Resource_Low Resolution/train_dfdhr_manifest_wilddeepfake.py' --repo_root '/home/jovyan/Deepfake_Low Resource_Low Resolution/DFD-HR' --metadata_csv '/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_w

In [ ]:
from pathlib import Path
import subprocess
import pandas as pd
import time

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")
OUT_DIR = PROJECT_ROOT / "outputs_dfdhr_wilddeepfake_train_no_amp_cc"

LOG_PATH = OUT_DIR / "train_dfdhr_wilddeepfake_resume_to_epoch5.log"
PID_PATH = OUT_DIR / "train_dfdhr_wilddeepfake_resume_to_epoch5.pid"

print("=" * 100)
print("DFD-HR resume-to-epoch-5 status")
print("=" * 100)

if PID_PATH.exists():
    pid = PID_PATH.read_text().strip()
    print("PID:", pid)
    print(subprocess.run(
        ["bash", "-lc", f"ps -p {pid} -o pid,etime,stat,%cpu,%mem,cmd || true"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    ).stdout)
else:
    print("No resume PID file found.")

print("\nGPU:")
print(subprocess.run(
    ["bash", "-lc", "nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv,noheader,nounits || true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
).stdout)

print("\nTraining history:")
hist_path = OUT_DIR / "training_history.csv"
if hist_path.exists():
    hist = pd.read_csv(hist_path)
    display(hist.tail(10))
else:
    print("No training_history.csv found.")

print("\nLog tail:")
if LOG_PATH.exists():
    lines = LOG_PATH.read_text(errors="replace").splitlines()
    for line in lines[-180:]:
        print(line)
else:
    print("No resume log found yet.")

In [2]:
!pip install torch scikit-learn torchvision pillow pandas transformers einops

  Using cached torch-2.13.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (38 kB)
  Using cached scikit_learn-1.9.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached torchvision-0.28.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (5.6 kB)
  Using cached pillow-12.3.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.1 kB)
  Using cached transformers-5.16.1-py3-none-any.whl.metadata (32 kB)
  Using cached einops-0.8.2-py3-none-any.whl.metadata (13 kB)
  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached cuda_toolkit-13.0.3.0-py2.py3-none-any.whl.metadata (17 kB)
  Using cached cuda_bindings-13.3.1-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.5 kB)
  Using cached nvidia_cudnn_c

In [4]:
# ============================================================
# Recover / estimate training time WITHOUT retraining
#
# Looks for:
#   1. per-epoch timing columns in training_history.csv
#   2. timestamped log lines
#   3. output file modification-time range as fallback estimate
#
# Saves:
#   outputs_training_time_summary/training_time_summary.csv
# ============================================================

from pathlib import Path
import os
import re
import pandas as pd
from datetime import datetime

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")

RUNS = {
    # "DFD-HR WildDeepfake": PROJECT_ROOT / "outputs_dfdhr_wilddeepfake_train_no_amp_cc",
    # "GenD WildDeepfake": PROJECT_ROOT / "outputs_gend_wilddeepfake_train",
    # "A-GRFN WildDeepfake": PROJECT_ROOT / "outputs_agrfn_wilddeepfake_ablation/E0_late_fusion",
    "FreqDebias WildDeepfake": PROJECT_ROOT / "outputs_freqdebias_wilddeepfake_train",
}

OUT_DIR = PROJECT_ROOT / "outputs_training_time_summary"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def seconds_to_hms(seconds):
    if seconds is None or pd.isna(seconds):
        return ""
    seconds = float(seconds)
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def find_files(run_dir):
    if not run_dir.exists():
        return []
    files = []
    for p in run_dir.rglob("*"):
        if p.is_file():
            files.append(p)
    return files


def estimate_from_file_mtimes(run_dir):
    files = find_files(run_dir)
    if not files:
        return None, None, None

    # Prefer files generated by training, not later evaluation files.
    useful_tokens = [
        "checkpoint", "best", "latest", "history", "train", "val",
        "log", "manifest", "metrics"
    ]

    useful = []
    for p in files:
        name = p.name.lower()
        if any(tok in name for tok in useful_tokens):
            useful.append(p)

    if not useful:
        useful = files

    mtimes = [(p, p.stat().st_mtime) for p in useful]
    earliest_p, earliest_t = min(mtimes, key=lambda x: x[1])
    latest_p, latest_t = max(mtimes, key=lambda x: x[1])

    duration = max(0.0, latest_t - earliest_t)

    return {
        "estimated_start_file": str(earliest_p),
        "estimated_end_file": str(latest_p),
        "estimated_start_time": datetime.fromtimestamp(earliest_t).isoformat(sep=" ", timespec="seconds"),
        "estimated_end_time": datetime.fromtimestamp(latest_t).isoformat(sep=" ", timespec="seconds"),
    }, duration, len(useful)


def estimate_from_history(run_dir):
    candidates = [
        run_dir / "training_history.csv",
        run_dir / "history.csv",
        run_dir / "training_period_results.csv",
        run_dir / "train_history.csv",
    ]

    for p in candidates:
        if not p.exists():
            continue

        try:
            df = pd.read_csv(p)
        except Exception:
            continue

        if len(df) == 0:
            continue

        cols = list(df.columns)
        lower_map = {c.lower(): c for c in cols}

        # Look for direct duration columns.
        duration_like = []
        for c in cols:
            lc = c.lower()
            if any(tok in lc for tok in ["epoch_time", "duration", "elapsed", "train_time", "time_sec", "seconds"]):
                if pd.api.types.is_numeric_dtype(df[c]):
                    duration_like.append(c)

        # Avoid accidentally summing things like "timestamp" encoded weirdly.
        for c in duration_like:
            vals = pd.to_numeric(df[c], errors="coerce").dropna()
            if len(vals) > 0 and vals.max() > 0:
                return {
                    "history_file": str(p),
                    "history_time_column": c,
                    "history_rows": len(df),
                }, float(vals.sum())

        # Look for start/end timestamp columns.
        timestamp_cols = []
        for c in cols:
            lc = c.lower()
            if any(tok in lc for tok in ["timestamp", "datetime", "date_time", "wall_time", "created_at"]):
                timestamp_cols.append(c)

        for c in timestamp_cols:
            ts = pd.to_datetime(df[c], errors="coerce").dropna()
            if len(ts) >= 2:
                duration = (ts.max() - ts.min()).total_seconds()
                if duration > 0:
                    return {
                        "history_file": str(p),
                        "history_time_column": c,
                        "history_rows": len(df),
                    }, float(duration)

    return None, None


def estimate_from_logs(run_dir):
    log_files = sorted(list(run_dir.rglob("*.log")))
    if not log_files:
        return None, None

    # Common timestamp formats.
    patterns = [
        r"\d{4}-\d{2}-\d{2}[ T]\d{2}:\d{2}:\d{2}",
        r"\d{2}/\d{2}/\d{4}[ T]\d{2}:\d{2}:\d{2}",
    ]

    all_times = []
    used_logs = []

    for log in log_files:
        try:
            text = log.read_text(errors="replace")
        except Exception:
            continue

        found_here = []
        for pat in patterns:
            for m in re.findall(pat, text):
                found_here.append(m)

        for item in found_here:
            parsed = pd.to_datetime(item, errors="coerce")
            if not pd.isna(parsed):
                all_times.append(parsed)
                used_logs.append(str(log))

    if len(all_times) >= 2:
        start = min(all_times)
        end = max(all_times)
        duration = (end - start).total_seconds()
        if duration > 0:
            return {
                "log_files_with_timestamps": "; ".join(sorted(set(used_logs))),
                "log_start_time": str(start),
                "log_end_time": str(end),
            }, float(duration)

    return None, None


rows = []

for model_name, run_dir in RUNS.items():
    row = {
        "Model": model_name,
        "Run Dir": str(run_dir),
        "Run Dir Exists": run_dir.exists(),
        "Best Available Source": "",
        "Training Time Seconds": None,
        "Training Time HH:MM:SS": "",
        "Reliability": "",
        "Notes": "",
    }

    if not run_dir.exists():
        row["Notes"] = "Output directory not found."
        rows.append(row)
        continue

    hist_meta, hist_sec = estimate_from_history(run_dir)
    log_meta, log_sec = estimate_from_logs(run_dir)
    file_meta, file_sec, n_files = estimate_from_file_mtimes(run_dir)

    if hist_sec is not None:
        row["Best Available Source"] = "training_history timing column"
        row["Training Time Seconds"] = hist_sec
        row["Reliability"] = "High"
        row["Notes"] = str(hist_meta)

    elif log_sec is not None:
        row["Best Available Source"] = "log timestamps"
        row["Training Time Seconds"] = log_sec
        row["Reliability"] = "Medium to high"
        row["Notes"] = str(log_meta)

    elif file_sec is not None:
        row["Best Available Source"] = "output file modification times"
        row["Training Time Seconds"] = file_sec
        row["Reliability"] = "Approximate"
        row["Notes"] = (
            "Estimated from earliest/latest relevant output file mtimes. "
            "This can be inaccurate if files were edited, copied, or evaluation outputs were added later. "
            f"Files considered: {n_files}. "
            f"{file_meta}"
        )

    else:
        row["Notes"] = "No usable history, logs, or output file timestamps found."

    if row["Training Time Seconds"] is not None:
        row["Training Time HH:MM:SS"] = seconds_to_hms(row["Training Time Seconds"])

    rows.append(row)

summary_df = pd.DataFrame(rows)
summary_path = OUT_DIR / "training_time_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("Saved:", summary_path)
display(summary_df)

Saved: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_training_time_summary/training_time_summary.csv


,Model,Run Dir,Run Dir Exists,Best Available Source,Training Time Seconds,Training Time HH:MM:SS,Reliability,Notes
0,FreqDebias WildDeepfake,/home/jovyan/Deepfake_Low Resource_Low Resolut...,True,output file modification times,10447.151007,02:54:07,Approximate,Estimated from earliest/latest relevant output...


In [3]:
# ============================================================
# DFD-HR FINAL IN-DOMAIN TEST
# Train: WildDeepfake
# Test: WildDeepfake test split
#
# Uses best_checkpoint.pt from the interrupted training folder.
# Does NOT train.
# ============================================================

from pathlib import Path
import sys
import os
import time
import yaml
import numpy as np
import pandas as pd
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_fscore_support,
    roc_curve,
)

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")
DFDHR_ROOT = PROJECT_ROOT / "DFD-HR"

TRAIN_OUT = PROJECT_ROOT / "outputs_dfdhr_wilddeepfake_train_no_amp_cc"

# Validation-selected checkpoint.
CKPT_PATH = TRAIN_OUT / "best_checkpoint.pt"

# In-domain WildDeepfake test split.
TEST_MANIFEST = PROJECT_ROOT / "outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv"

OUT_DIR = PROJECT_ROOT / "outputs_dfdhr_wilddeepfake_indomain_final_test_best"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT = "test"
IMAGE_SIZE = 448
FRAMES_PER_VIDEO = 32
BATCH_SIZE = 4
NUM_WORKERS = 0
UNCERTAINTY_CONFIDENCE_THRESHOLD = 0.60
SEED = 42

if not CKPT_PATH.exists():
    raise FileNotFoundError(f"Missing checkpoint: {CKPT_PATH}")

if not TEST_MANIFEST.exists():
    raise FileNotFoundError(f"Missing test manifest: {TEST_MANIFEST}")

print("=" * 100)
print("DFD-HR final in-domain test: WildDeepfake -> WildDeepfake")
print("=" * 100)
print("Checkpoint:", CKPT_PATH)
print("Test manifest:", TEST_MANIFEST)
print("Output dir:", OUT_DIR)


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def load_yaml(path):
    path = Path(path)
    if not path.exists():
        return {}
    with open(path, "r") as f:
        return yaml.safe_load(f) or {}


def normalize_manifest(path, split="test"):
    df = pd.read_csv(path)

    if "frame_path" not in df.columns:
        if "image_path" in df.columns:
            df["frame_path"] = df["image_path"].astype(str)
        else:
            raise ValueError("Manifest needs frame_path or image_path.")

    if "video_id" not in df.columns:
        if "sample_unit_id" in df.columns:
            df["video_id"] = df["sample_unit_id"].astype(str)
        elif "group_id" in df.columns:
            df["video_id"] = df["group_id"].astype(str)
        else:
            df["video_id"] = df["frame_path"].apply(lambda x: str(Path(x).parent))

    required = ["frame_path", "split", "label", "video_id"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing manifest columns: {missing}")

    df["frame_path"] = df["frame_path"].astype(str)
    df["video_id"] = df["video_id"].astype(str)
    df["split"] = df["split"].astype(str)
    df["label"] = df["label"].astype(int)

    df = df[df["split"] == split].copy()

    if len(df) == 0:
        raise RuntimeError(f"No rows found for split={split}")

    return df


def sample_frames_per_video(df, frames_per_video=32):
    parts = []

    for vid, g in df.groupby("video_id", sort=False):
        g = g.sort_values("frame_path").reset_index(drop=True)

        if len(g) <= frames_per_video:
            parts.append(g)
        else:
            idx = np.linspace(0, len(g) - 1, frames_per_video).round().astype(int)
            parts.append(g.iloc[idx].copy())

    out = pd.concat(parts, ignore_index=True)
    out = out.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    return out


class ManifestImageDataset(Dataset):
    def __init__(self, df, image_size=448):
        self.df = df.reset_index(drop=True)
        self.image_size = image_size
        self.tf = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row["frame_path"]
        label = int(row["label"])
        video_id = str(row["video_id"])

        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            img = Image.new("RGB", (self.image_size, self.image_size))

        return {
            "image": self.tf(img),
            "label": torch.tensor(label, dtype=torch.long),
            "path": path,
            "video_id": video_id,
        }


def extract_logits(output):
    if torch.is_tensor(output):
        return output

    if isinstance(output, dict):
        for key in [
            "logits", "cls", "pred", "prediction", "output", "out",
            "y_pred", "final_output"
        ]:
            if key in output and torch.is_tensor(output[key]):
                return output[key]

        if "prob" in output and torch.is_tensor(output["prob"]):
            prob = output["prob"]
            if prob.ndim == 1:
                return torch.logit(prob.clamp(1e-6, 1.0 - 1e-6)).view(-1, 1)
            if prob.ndim == 2 and prob.shape[1] == 1:
                return torch.logit(prob[:, 0].clamp(1e-6, 1.0 - 1e-6)).view(-1, 1)
            if prob.ndim == 2 and prob.shape[1] >= 2:
                return torch.log(prob[:, :2].clamp(1e-6, 1.0))

        for value in output.values():
            if torch.is_tensor(value) and value.ndim >= 1:
                return value

    if isinstance(output, (list, tuple)):
        for value in output:
            if torch.is_tensor(value):
                return value

    raise RuntimeError(f"Could not extract logits from output type: {type(output)}")


def forward_model(model, images, labels):
    data_dict = {
        "image": images,
        "label": labels,
        "mask": None,
        "landmark": None,
    }

    try:
        output = model(data_dict, inference=True)
    except TypeError:
        output = model(data_dict)

    logits = extract_logits(output)

    if logits.ndim == 1:
        logits = logits.view(-1, 1)

    if logits.ndim > 2:
        logits = logits.view(logits.shape[0], -1)

    return logits


def fake_scores_from_logits(logits):
    if logits.shape[1] == 1:
        return torch.sigmoid(logits[:, 0])
    return torch.softmax(logits[:, :2], dim=1)[:, 1]


def compute_eer(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)

    if len(np.unique(y_true)) < 2:
        return np.nan

    fpr, tpr, _ = roc_curve(y_true, y_score)
    fnr = 1.0 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    return float((fpr[idx] + fnr[idx]) / 2.0)


def compute_uncertain_rate(y_score, threshold=0.60):
    y_score = np.asarray(y_score).astype(float)
    confidence = np.maximum(y_score, 1.0 - y_score)
    return float(np.mean(confidence < threshold) * 100.0)


def compute_metric_row(y_true, y_score, level, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    y_pred = (y_score >= threshold).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1],
        zero_division=0,
    )

    return {
        "Level": level,
        "Decision Threshold": threshold,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "AUPRC": average_precision_score(y_true, y_score) if len(np.unique(y_true)) == 2 else np.nan,
        "AUCROC": roc_auc_score(y_true, y_score) if len(np.unique(y_true)) == 2 else np.nan,
        "EER (Equal Error Rate)": compute_eer(y_true, y_score),
        "Precision (0)": precision[0],
        "Recall (0)": recall[0],
        "F1 (0)": f1[0],
        "Precision (1)": precision[1],
        "Recall (1)": recall[1],
        "F1 (1)": f1[1],
        "Uncertain Rate (%)": compute_uncertain_rate(
            y_score,
            threshold=UNCERTAINTY_CONFIDENCE_THRESHOLD,
        ),
        "Predicted Real Count": int((y_pred == 0).sum()),
        "Predicted Fake Count": int((y_pred == 1).sum()),
        "Rows": int(len(y_true)),
    }


# ------------------------------------------------------------
# Load WildDeepfake test split
# ------------------------------------------------------------

test_df = normalize_manifest(TEST_MANIFEST, split=SPLIT)
test_df = sample_frames_per_video(test_df, frames_per_video=FRAMES_PER_VIDEO)
test_df.to_csv(OUT_DIR / "test_manifest_used.csv", index=False)

print("\nWildDeepfake test rows after sampling:", len(test_df))
print("WildDeepfake test videos:", test_df["video_id"].nunique())
print("\nTest counts:")
display(pd.crosstab(test_df["split"], test_df["label"]))


# ------------------------------------------------------------
# Load DFD-HR model
# ------------------------------------------------------------

sys.path.insert(0, str(DFDHR_ROOT / "training"))
from detectors import DETECTOR

detector_cfg = load_yaml(DFDHR_ROOT / "training/config/detector/dfd_hr.yaml")
test_cfg = load_yaml(DFDHR_ROOT / "training/config/test_config.yaml")

config = {}
config.update(test_cfg)
config.update(detector_cfg)
config["cuda"] = torch.cuda.is_available()
config["manualSeed"] = SEED
config["test_batchSize"] = BATCH_SIZE
config["workers"] = NUM_WORKERS

detector_name = config.get("name", config.get("model_name", "dfd_hr"))
config["model_name"] = detector_name

try:
    model_cls = DETECTOR[detector_name]
except Exception:
    model_cls = DETECTOR["dfd_hr"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nDevice:", device)

model = model_cls(config).to(device)

# PyTorch 2.6 compatibility: local trusted checkpoint.
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)

state = None
for key in ["model", "model_state_dict", "state_dict", "net", "network"]:
    if isinstance(ckpt, dict) and key in ckpt and isinstance(ckpt[key], dict):
        state = ckpt[key]
        print("Using checkpoint state key:", key)
        break

if state is None:
    state = ckpt

missing, unexpected = model.load_state_dict(state, strict=False)

print("\nLoaded DFD-HR checkpoint.")
print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

if len(missing) > 0:
    print("First missing keys:", missing[:10])
if len(unexpected) > 0:
    print("First unexpected keys:", unexpected[:10])

model.eval()


# ------------------------------------------------------------
# DataLoader
# ------------------------------------------------------------

dataset = ManifestImageDataset(test_df, image_size=IMAGE_SIZE)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

print("\nEvaluation batches:", len(loader))
print("Evaluation images:", len(dataset))
print("Batch size:", BATCH_SIZE)


# ------------------------------------------------------------
# First batch check
# ------------------------------------------------------------

print("\nRunning first-batch forward check...")

first_batch = next(iter(loader))
first_images = first_batch["image"].to(device)
first_labels = first_batch["label"].to(device)

with torch.inference_mode():
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    first_logits = forward_model(model, first_images, first_labels)
    first_scores = fake_scores_from_logits(first_logits)
    if device.type == "cuda":
        torch.cuda.synchronize()
    first_elapsed = time.time() - t0

print("First batch logits shape:", tuple(first_logits.shape))
print("First batch scores shape:", tuple(first_scores.shape))
print("First batch elapsed sec:", round(first_elapsed, 4))
print("First batch p_fake range:", float(first_scores.min()), "to", float(first_scores.max()))


# ------------------------------------------------------------
# Evaluate
# ------------------------------------------------------------

all_rows = []
start_total = time.time()
num_images = 0

iterator = loader
if tqdm is not None:
    iterator = tqdm(loader, desc="Evaluating DFD-HR WildDeepfake final test", total=len(loader))

with torch.inference_mode():
    for batch_idx, batch in enumerate(iterator, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        if device.type == "cuda":
            torch.cuda.synchronize()

        t0 = time.time()
        logits = forward_model(model, images, labels)

        if device.type == "cuda":
            torch.cuda.synchronize()

        elapsed = time.time() - t0

        scores = fake_scores_from_logits(logits).detach().cpu().numpy()
        labels_np = labels.detach().cpu().numpy()

        paths = batch["path"]
        vids = batch["video_id"]

        bs = len(labels_np)
        num_images += bs

        for i in range(bs):
            p_fake = float(scores[i])
            confidence = max(p_fake, 1.0 - p_fake)
            pred = int(p_fake >= 0.5)
            uncertain = int(confidence < UNCERTAINTY_CONFIDENCE_THRESHOLD)

            all_rows.append({
                "image_path": paths[i],
                "video_id": vids[i],
                "label": int(labels_np[i]),
                "p_fake": p_fake,
                "prediction": pred,
                "confidence": confidence,
                "uncertain": uncertain,
                "inference_time_sec_per_image_batch_avg": elapsed / max(bs, 1),
            })

        if batch_idx % 100 == 0:
            partial_path = OUT_DIR / "test_frame_predictions_partial.csv"
            pd.DataFrame(all_rows).to_csv(partial_path, index=False)
            elapsed_total = time.time() - start_total
            print(
                f"\nProgress batch {batch_idx}/{len(loader)} | "
                f"images={num_images}/{len(dataset)} | "
                f"images/sec={num_images / max(elapsed_total, 1e-9):.3f}"
            )

total_elapsed = time.time() - start_total

frame_pred_df = pd.DataFrame(all_rows)
frame_pred_df.to_csv(OUT_DIR / "test_frame_predictions.csv", index=False)


# ------------------------------------------------------------
# Video aggregation
# ------------------------------------------------------------

video_pred_df = (
    frame_pred_df
    .groupby("video_id", as_index=False)
    .agg(
        label=("label", "first"),
        p_fake=("p_fake", "mean"),
        frame_count=("p_fake", "size"),
    )
)

video_pred_df["prediction"] = (video_pred_df["p_fake"] >= 0.5).astype(int)
video_pred_df["confidence"] = np.maximum(video_pred_df["p_fake"], 1.0 - video_pred_df["p_fake"])
video_pred_df["uncertain"] = (
    video_pred_df["confidence"] < UNCERTAINTY_CONFIDENCE_THRESHOLD
).astype(int)

video_pred_df.to_csv(OUT_DIR / "test_video_predictions.csv", index=False)


# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

frame_metrics = compute_metric_row(
    frame_pred_df["label"].values,
    frame_pred_df["p_fake"].values,
    level="Frame",
)

video_metrics = compute_metric_row(
    video_pred_df["label"].values,
    video_pred_df["p_fake"].values,
    level="Video",
)

metrics_df = pd.DataFrame([frame_metrics, video_metrics])
metrics_df.to_csv(OUT_DIR / "test_metrics.csv", index=False)

time_df = pd.DataFrame([{
    "num_images": int(num_images),
    "total_inference_time_sec": total_elapsed,
    "avg_inference_time_sec_per_image": total_elapsed / max(num_images, 1),
    "images_per_second": num_images / max(total_elapsed, 1e-9),
    "batch_size": BATCH_SIZE,
    "image_size": IMAGE_SIZE,
    "frames_per_video": FRAMES_PER_VIDEO,
    "checkpoint": str(CKPT_PATH),
    "test_manifest": str(TEST_MANIFEST),
    "output_dir": str(OUT_DIR),
}])
time_df.to_csv(OUT_DIR / "test_inference_time.csv", index=False)


# ------------------------------------------------------------
# Print final results
# ------------------------------------------------------------

print("\nFinished DFD-HR final in-domain test.")
print("Total images:", num_images)
print("Total elapsed sec:", round(total_elapsed, 2))
print("Images/sec:", round(num_images / max(total_elapsed, 1e-9), 4))

print("\nSaved:")
print(OUT_DIR / "test_metrics.csv")
print(OUT_DIR / "test_frame_predictions.csv")
print(OUT_DIR / "test_video_predictions.csv")
print(OUT_DIR / "test_inference_time.csv")

print("\nDFD-HR final in-domain WildDeepfake test metrics:")
display(metrics_df)

print("\nInference time:")
display(time_df)

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DFD-HR final in-domain test: WildDeepfake -> WildDeepfake
Checkpoint: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wilddeepfake_train_no_amp_cc/best_checkpoint.pt
Test manifest: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv
Output dir: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wilddeepfake_indomain_final_test_best

WildDeepfake test rows after sampling: 25792
WildDeepfake test videos: 806

Test counts:


label,0,1
split,,
test,12672,13120



Device: cuda


Loading weights: 100%|██████████| 590/590 [00:00<00:00, 10519.28it/s]


Using checkpoint state key: model

Loaded DFD-HR checkpoint.
Missing keys: 0
Unexpected keys: 0

Evaluation batches: 6448
Evaluation images: 25792
Batch size: 4

Running first-batch forward check...


... (131 outputs left out)


Finished DFD-HR final in-domain test.
Total images: 25792
Total elapsed sec: 5103.92
Images/sec: 5.0534

Saved:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wilddeepfake_indomain_final_test_best/test_metrics.csv
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wilddeepfake_indomain_final_test_best/test_frame_predictions.csv
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wilddeepfake_indomain_final_test_best/test_video_predictions.csv
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wilddeepfake_indomain_final_test_best/test_inference_time.csv

DFD-HR final in-domain WildDeepfake test metrics:


,Level,Decision Threshold,Accuracy,Balanced Accuracy,AUPRC,AUCROC,EER (Equal Error Rate),Precision (0),Recall (0),F1 (0),Precision (1),Recall (1),F1 (1),Uncertain Rate (%),Predicted Real Count,Predicted Fake Count,Rows
0,Frame,0.5,0.830994,0.831597,0.897398,0.897184,0.172804,0.804662,0.866319,0.834353,0.860565,0.796875,0.827496,2.268145,13643,12149,25792
1,Video,0.5,0.862283,0.862823,0.932167,0.925684,0.143921,0.836879,0.893939,0.864469,0.890339,0.831707,0.860025,6.079404,423,383,806



Inference time:


,num_images,total_inference_time_sec,avg_inference_time_sec_per_image,images_per_second,batch_size,image_size,frames_per_video,checkpoint,test_manifest,output_dir
0,25792,5103.924222,0.197888,5.053367,4,448,32,/home/jovyan/Deepfake_Low Resource_Low Resolut...,/home/jovyan/Deepfake_Low Resource_Low Resolut...,/home/jovyan/Deepfake_Low Resource_Low Resolut...


In [4]:
# ============================================================
# DFD-HR training time summary
# Uses completed epochs from training_history.csv
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import time

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")
OUT_DIR = PROJECT_ROOT / "outputs_dfdhr_wilddeepfake_train_no_amp_cc"

HISTORY_PATH = OUT_DIR / "training_history.csv"
SUMMARY_PATH = OUT_DIR / "training_time_summary.csv"

if not HISTORY_PATH.exists():
    raise FileNotFoundError(f"Missing training history: {HISTORY_PATH}")

hist = pd.read_csv(HISTORY_PATH)

print("Training history path:", HISTORY_PATH)
display(hist)

# ------------------------------------------------------------
# Detect columns
# ------------------------------------------------------------

epoch_col = "epoch" if "epoch" in hist.columns else ("Epoch" if "Epoch" in hist.columns else None)

if epoch_col is None:
    raise RuntimeError(f"Could not find epoch column. Columns are: {list(hist.columns)}")

elapsed_candidates = [
    "elapsed_min",
    "epoch_elapsed_min",
    "train_elapsed_min",
    "time_min",
    "elapsed_minutes",
]

elapsed_col = None
for c in elapsed_candidates:
    if c in hist.columns:
        elapsed_col = c
        break

if elapsed_col is None:
    raise RuntimeError(
        "Could not find elapsed time column. "
        f"Looked for {elapsed_candidates}. Columns are: {list(hist.columns)}"
    )

hist[epoch_col] = pd.to_numeric(hist[epoch_col], errors="coerce")
hist[elapsed_col] = pd.to_numeric(hist[elapsed_col], errors="coerce")

hist_clean = hist.dropna(subset=[epoch_col, elapsed_col]).copy()
hist_clean = hist_clean.sort_values(epoch_col)

completed_epochs = int(hist_clean[epoch_col].max())
num_completed_rows = len(hist_clean)

# ------------------------------------------------------------
# Determine whether elapsed column is per-epoch or cumulative
# ------------------------------------------------------------
# If elapsed_min is strictly increasing and final is much larger than first,
# it may be cumulative. Otherwise, treat it as per-epoch.
# In your DFD-HR history, elapsed_min values are similar per epoch,
# so this should be treated as per-epoch duration.

elapsed_values = hist_clean[elapsed_col].values.astype(float)

is_monotonic = np.all(np.diff(elapsed_values) >= 0)
looks_cumulative = bool(
    is_monotonic and
    len(elapsed_values) >= 2 and
    elapsed_values[-1] > elapsed_values[0] * 1.25
)

if looks_cumulative:
    total_completed_min = float(elapsed_values[-1])
    epoch_time_min_values = np.diff(np.concatenate([[0.0], elapsed_values]))
    elapsed_interpretation = "cumulative_elapsed_minutes"
else:
    total_completed_min = float(np.nansum(elapsed_values))
    epoch_time_min_values = elapsed_values
    elapsed_interpretation = "per_epoch_elapsed_minutes"

avg_epoch_min = float(np.nanmean(epoch_time_min_values))
median_epoch_min = float(np.nanmedian(epoch_time_min_values))

# ------------------------------------------------------------
# Best epoch from validation AUCROC or balanced accuracy
# ------------------------------------------------------------

best_metric_col = None
for c in ["val_AUCROC", "val_Balanced Accuracy", "val_Accuracy", "AUCROC", "Balanced Accuracy", "Accuracy"]:
    if c in hist_clean.columns:
        best_metric_col = c
        break

if best_metric_col is not None:
    hist_clean[best_metric_col] = pd.to_numeric(hist_clean[best_metric_col], errors="coerce")
    best_idx = hist_clean[best_metric_col].idxmax()
    best_epoch = int(hist_clean.loc[best_idx, epoch_col])
    best_metric_value = float(hist_clean.loc[best_idx, best_metric_col])
else:
    best_epoch = None
    best_metric_value = np.nan

# ------------------------------------------------------------
# Checkpoint file info
# ------------------------------------------------------------

def file_info(path):
    path = Path(path)
    if not path.exists():
        return {
            "exists": False,
            "size_mb": np.nan,
            "age_min": np.nan,
            "modified_time": None,
        }

    return {
        "exists": True,
        "size_mb": path.stat().st_size / 1024**2,
        "age_min": (time.time() - path.stat().st_mtime) / 60,
        "modified_time": time.ctime(path.stat().st_mtime),
    }

latest_info = file_info(OUT_DIR / "latest_checkpoint.pt")
best_info = file_info(OUT_DIR / "best_checkpoint.pt")

# ------------------------------------------------------------
# Summary table
# ------------------------------------------------------------

summary = {
    "model": "DFD-HR",
    "train_dataset": "WildDeepfake",
    "training_output_dir": str(OUT_DIR),
    "history_file": str(HISTORY_PATH),
    "completed_epochs": completed_epochs,
    "completed_epoch_rows": num_completed_rows,
    "elapsed_column": elapsed_col,
    "elapsed_interpretation": elapsed_interpretation,
    "total_completed_training_time_min": total_completed_min,
    "total_completed_training_time_hours": total_completed_min / 60.0,
    "avg_time_per_completed_epoch_min": avg_epoch_min,
    "avg_time_per_completed_epoch_hours": avg_epoch_min / 60.0,
    "median_time_per_completed_epoch_min": median_epoch_min,
    "median_time_per_completed_epoch_hours": median_epoch_min / 60.0,
    "best_epoch_by_validation": best_epoch,
    "best_metric_column": best_metric_col,
    "best_metric_value": best_metric_value,
    "latest_checkpoint_exists": latest_info["exists"],
    "latest_checkpoint_size_mb": latest_info["size_mb"],
    "latest_checkpoint_modified_time": latest_info["modified_time"],
    "best_checkpoint_exists": best_info["exists"],
    "best_checkpoint_size_mb": best_info["size_mb"],
    "best_checkpoint_modified_time": best_info["modified_time"],
    "note": (
        "Training time is computed from completed epochs in training_history.csv. "
        "Interrupted, incomplete epochs and server downtime are not counted unless they produced a completed history row."
    ),
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv(SUMMARY_PATH, index=False)

print("\nTraining time summary saved to:")
print(SUMMARY_PATH)

print("\nTraining time summary:")
display(summary_df)

# ------------------------------------------------------------
# Per-epoch timing table
# ------------------------------------------------------------

per_epoch_df = hist_clean[[epoch_col, elapsed_col]].copy()
per_epoch_df = per_epoch_df.rename(columns={
    epoch_col: "epoch",
    elapsed_col: "recorded_elapsed_min",
})

per_epoch_df["interpreted_epoch_time_min"] = epoch_time_min_values
per_epoch_df["interpreted_epoch_time_hours"] = per_epoch_df["interpreted_epoch_time_min"] / 60.0

per_epoch_path = OUT_DIR / "training_time_per_epoch.csv"
per_epoch_df.to_csv(per_epoch_path, index=False)

print("\nPer-epoch timing saved to:")
print(per_epoch_path)

display(per_epoch_df)

Training history path: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wilddeepfake_train_no_amp_cc/training_history.csv


,epoch,train_loss,train_acc,val_loss,val_Accuracy,val_Balanced Accuracy,val_AUPRC,val_AUCROC,val_Precision (0),val_Recall (0),val_F1 (0),val_Precision (1),val_Recall (1),val_F1 (1),elapsed_min
0,1,0.105614,0.960710,0.166057,0.957133,0.956557,0.984768,0.987476,0.950454,0.968658,0.959470,0.964782,0.944456,0.954511,1577.744576
1,2,0.050913,0.982132,0.182802,0.949501,0.949262,0.984005,0.985173,0.949571,0.954271,0.951915,0.949422,0.944254,0.946831,1577.577095
2,3,0.041971,0.985680,0.177250,0.951133,0.951957,0.985823,0.986002,0.970963,0.934659,0.952465,0.930964,0.969254,0.949723,1579.990839



Training time summary saved to:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wilddeepfake_train_no_amp_cc/training_time_summary.csv

Training time summary:


,model,train_dataset,training_output_dir,history_file,completed_epochs,completed_epoch_rows,elapsed_column,elapsed_interpretation,total_completed_training_time_min,total_completed_training_time_hours,...,best_epoch_by_validation,best_metric_column,best_metric_value,latest_checkpoint_exists,latest_checkpoint_size_mb,latest_checkpoint_modified_time,best_checkpoint_exists,best_checkpoint_size_mb,best_checkpoint_modified_time,note
0,DFD-HR,WildDeepfake,/home/jovyan/Deepfake_Low Resource_Low Resolut...,/home/jovyan/Deepfake_Low Resource_Low Resolut...,3,3,elapsed_min,per_epoch_elapsed_minutes,4735.31251,78.921875,...,1,val_AUCROC,0.987476,True,376.03125,Tue Sep 1 04:24:24 2026,True,2351.041071,Sat Aug 29 19:29:17 2026,Training time is computed from completed epoch...



Per-epoch timing saved to:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wilddeepfake_train_no_amp_cc/training_time_per_epoch.csv


,epoch,recorded_elapsed_min,interpreted_epoch_time_min,interpreted_epoch_time_hours
0,1,1577.744576,1577.744576,26.295743
1,2,1577.577095,1577.577095,26.292952
2,3,1579.990839,1579.990839,26.333181


In [2]:
# ============================================================
# Create FreqDebias-safe 32fpv WildDeepfake manifest
# Then launch FreqDebias training with nohup
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import subprocess
import shlex

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")
WILD_OUT = PROJECT_ROOT / "outputs_wilddeepfake"

CANONICAL_MANIFEST = WILD_OUT / "wilddeepfake_manifest_canonical.csv"
SAFE_MANIFEST = WILD_OUT / "wilddeepfake_manifest_canonical_freqdebias_safe.csv"
SAMPLED_MANIFEST = WILD_OUT / "wilddeepfake_manifest_canonical_freqdebias_safe_32fpv.csv"

OUT_DIR = PROJECT_ROOT / "outputs_freqdebias_wilddeepfake_train_32fpv"
LOG_PATH = OUT_DIR / "train_freqdebias_nohup.log"
PID_PATH = OUT_DIR / "train_freqdebias_nohup.pid"

OUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Create safe full manifest if needed
# ------------------------------------------------------------

if not SAFE_MANIFEST.exists():
    print("Creating safe FreqDebias manifest from canonical manifest...")

    if not CANONICAL_MANIFEST.exists():
        raise FileNotFoundError(f"Missing canonical manifest: {CANONICAL_MANIFEST}")

    df = pd.read_csv(CANONICAL_MANIFEST)

    if "frame_path" not in df.columns:
        if "image_path" in df.columns:
            df["frame_path"] = df["image_path"].astype(str)
        else:
            raise ValueError("Canonical manifest needs frame_path or image_path.")

    # Critical: remove image_path because freqdebias_ffpp.py renames frame_path -> image_path internally.
    if "image_path" in df.columns:
        df = df.drop(columns=["image_path"])

    required = ["frame_path", "split", "label", "video_id"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df.to_csv(SAFE_MANIFEST, index=False)
    print("Saved:", SAFE_MANIFEST)
else:
    print("Safe manifest already exists:", SAFE_MANIFEST)

# ------------------------------------------------------------
# 2. Create 32 frames-per-video sampled manifest
# ------------------------------------------------------------

print("\nCreating 32fpv sampled manifest...")

df = pd.read_csv(SAFE_MANIFEST)

if "image_path" in df.columns:
    df = df.drop(columns=["image_path"])

required = ["frame_path", "split", "label", "video_id"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["frame_path"] = df["frame_path"].astype(str)
df["split"] = df["split"].astype(str)
df["label"] = df["label"].astype(int)
df["video_id"] = df["video_id"].astype(str)

print("Input rows:", len(df))
print("\nInput counts:")
display(pd.crosstab(df["split"], df["label"]))

FRAMES_PER_VIDEO = 32
SEED = 42

parts = []

for (split_name, label, video_id), g in df.groupby(["split", "label", "video_id"], sort=False):
    g = g.sort_values("frame_path").reset_index(drop=True)

    if len(g) <= FRAMES_PER_VIDEO:
        sampled = g
    else:
        idx = np.linspace(0, len(g) - 1, FRAMES_PER_VIDEO).round().astype(int)
        sampled = g.iloc[idx].copy()

    parts.append(sampled)

sampled_df = pd.concat(parts, ignore_index=True)
sampled_df = sampled_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

# Safety: FreqDebias should not receive image_path.
if "image_path" in sampled_df.columns:
    sampled_df = sampled_df.drop(columns=["image_path"])

sampled_df.to_csv(SAMPLED_MANIFEST, index=False)

print("\nSaved sampled manifest:")
print(SAMPLED_MANIFEST)

print("\nSampled rows:", len(sampled_df))
print("\nSampled counts:")
display(pd.crosstab(sampled_df["split"], sampled_df["label"]))

print("\nUnique videos:")
display(sampled_df.groupby(["split", "label"])["video_id"].nunique().unstack(fill_value=0))

print("\nMax frames per video:", sampled_df.groupby("video_id").size().max())

# ------------------------------------------------------------
# 3. Launch FreqDebias with nohup
# ------------------------------------------------------------

cmd = [
    "python", "-u", "freqdebias_ffpp.py",
    "--metadata_csv", str(SAMPLED_MANIFEST),
    "--out_dir", str(OUT_DIR),
    "--batch_size", "32",
    "--image_size", "256",
    "--num_workers", "4",
]

print("\n" + "=" * 100)
print("Checking existing FreqDebias processes")
print("=" * 100)

existing_freq = subprocess.run(
    ["bash", "-lc", "ps aux | grep -i 'freqdebias_ffpp.py' | grep -v grep || true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
).stdout.strip()

if existing_freq:
    print("FreqDebias is already running. Not launching duplicate:")
    print(existing_freq)
else:
    print("Launching FreqDebias WildDeepfake training")
    print("Command:")
    print(" ".join(shlex.quote(x) for x in cmd))
    print("Output:", OUT_DIR)
    print("Log:", LOG_PATH)

    bash_cmd = (
        "cd " + shlex.quote(str(PROJECT_ROOT)) + " && "
        "PYTHONUNBUFFERED=1 nohup " + " ".join(shlex.quote(x) for x in cmd) +
        " > " + shlex.quote(str(LOG_PATH)) + " 2>&1 & echo $!"
    )

    proc = subprocess.run(
        ["bash", "-lc", bash_cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    pid = proc.stdout.strip().splitlines()[-1]
    PID_PATH.write_text(pid)

    print("\nLaunched FreqDebias PID:", pid)
    print("PID saved to:", PID_PATH)
    print("Log:", LOG_PATH)

Safe manifest already exists: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_manifest_canonical_freqdebias_safe.csv

Creating 32fpv sampled manifest...
Input rows: 1180100

Input counts:


label,0,1
split,,
test,58659,107003
train,346013,563531
val,35864,69030



Saved sampled manifest:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_manifest_canonical_freqdebias_safe_32fpv.csv

Sampled rows: 234049

Sampled counts:


label,0,1
split,,
test,12672,13120
train,98177,89248
val,10912,9920



Unique videos:


label,0,1
split,,
test,396,410
train,3069,2789
val,341,310



Max frames per video: 32

Checking existing FreqDebias processes
Launching FreqDebias WildDeepfake training
Command:
python -u freqdebias_ffpp.py --metadata_csv '/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_manifest_canonical_freqdebias_safe_32fpv.csv' --out_dir '/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_freqdebias_wilddeepfake_train_32fpv' --batch_size 32 --image_size 256 --num_workers 4
Output: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_freqdebias_wilddeepfake_train_32fpv
Log: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_freqdebias_wilddeepfake_train_32fpv/train_freqdebias_nohup.log

Launched FreqDebias PID: 901
PID saved to: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_freqdebias_wilddeepfake_train_32fpv/train_freqdebias_nohup.pid
Log: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_freqdebias_wilddeepfake_train_32fpv/train_freqdebias_nohup.log


In [4]:
from pathlib import Path
import subprocess
import pandas as pd
import time

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")
OUT_DIR = PROJECT_ROOT / "outputs_freqdebias_wilddeepfake_train_32fpv"
LOG_PATH = OUT_DIR / "train_freqdebias_nohup.log"
PID_PATH = OUT_DIR / "train_freqdebias_nohup.pid"

if PID_PATH.exists():
    pid = PID_PATH.read_text().strip()
    print("PID:", pid)
    print(subprocess.run(
        ["bash", "-lc", f"ps -p {pid} -o pid,ppid,etime,stat,%cpu,%mem,rss,cmd || true"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    ).stdout)

print("\nGPU:")
print(subprocess.run(
    ["bash", "-lc", "nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv,noheader,nounits || true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
).stdout or "No GPU process found.")

print("\nOutput files:")
for name in [
    "best_checkpoint.pt",
    "latest_checkpoint.pt",
    "freqdebias_best.pt",
    "freqdebias_latest.pt",
    "training_history.csv",
    "training_period_results.csv",
    "test_metrics.csv",
    "test_frame_predictions.csv",
    "test_video_predictions.csv",
    "test_inference_time.csv",
    "train_freqdebias_nohup.log",
]:
    p = OUT_DIR / name
    if p.exists():
        print(f"FOUND   {name:35s} size_mb={p.stat().st_size/1024**2:.3f} age_min={(time.time()-p.stat().st_mtime)/60:.1f}")
    else:
        print(f"MISSING {name}")

print("\nLog tail:")
if LOG_PATH.exists():
    lines = LOG_PATH.read_text(errors="replace").splitlines()
    print("Log lines:", len(lines))
    for line in lines[-120:]:
        print(line)
else:
    print("No log yet.")

PID: 901
    PID    PPID     ELAPSED STAT %CPU %MEM   RSS CMD


GPU:
No GPU process found.

Output files:
MISSING best_checkpoint.pt
MISSING latest_checkpoint.pt
MISSING freqdebias_best.pt
FOUND   freqdebias_latest.pt                size_mb=257.347 age_min=6.4
FOUND   training_history.csv                size_mb=0.003 age_min=6.4
MISSING training_period_results.csv
FOUND   test_metrics.csv                    size_mb=0.001 age_min=0.0
FOUND   test_frame_predictions.csv          size_mb=4.772 age_min=0.0
FOUND   test_video_predictions.csv          size_mb=0.066 age_min=0.0
FOUND   test_inference_time.csv             size_mb=0.000 age_min=0.0
FOUND   train_freqdebias_nohup.log          size_mb=9.231 age_min=0.0

Log tail:
Log lines: 75497
Predict test: 100%|██████████| 806/806 [06:19<00:00,  2.13it/s]

Test metrics:
Level  Accuracy    AUPRC   AUCROC  EER (Equal Error Rate)  Precision (0)  Recall (0)   F1 (0)  Precision (1)  Recall (1)   F1 (1)  Uncertain Rate (%)
Frame  0.802458 0.875946 0

In [ ]:
from pathlib import Path
import subprocess
import shlex

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")

SCRIPT = PROJECT_ROOT / "DFD-HR/run_dfd_hr_df40.py"
MANIFEST = PROJECT_ROOT / "outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv"

TRAIN_OUT = PROJECT_ROOT / "outputs_dfdhr_wilddeepfake_train"
CKPT = TRAIN_OUT / "latest_checkpoint.pt"

OUT_DIR = PROJECT_ROOT / "outputs_dfdhr_wilddeepfake_latest_test"
LOG_PATH = OUT_DIR / "dfdhr_latest_test.log"
PID_PATH = OUT_DIR / "dfdhr_latest_test.pid"

OUT_DIR.mkdir(parents=True, exist_ok=True)

if not SCRIPT.exists():
    raise FileNotFoundError(SCRIPT)

if not MANIFEST.exists():
    raise FileNotFoundError(MANIFEST)

if not CKPT.exists():
    raise FileNotFoundError(f"Missing latest checkpoint: {CKPT}")

cmd = [
    "python", "-u", str(SCRIPT),
    "--repo_root", str(PROJECT_ROOT / "DFD-HR"),
    "--metadata_csv", str(MANIFEST),
    "--ckpt_path", str(CKPT),
    "--out_dir", str(OUT_DIR),
    "--split", "test",
    "--image_size", "448",
    "--frames_per_video", "32",
    "--batch_size", "8",
    "--num_workers", "4",
    "--path_col", "frame_path",
    "--label_col", "label",
    "--split_col", "split",
    "--group_col", "video_id",
    "--allow_non_strict",
]

print("Command:")
print(" ".join(shlex.quote(x) for x in cmd))
print("Checkpoint:", CKPT)
print("Output:", OUT_DIR)
print("Log:", LOG_PATH)

bash_cmd = (
    "cd " + shlex.quote(str(PROJECT_ROOT)) + " && "
    "PYTHONUNBUFFERED=1 nohup " + " ".join(shlex.quote(x) for x in cmd) +
    " > " + shlex.quote(str(LOG_PATH)) + " 2>&1 & echo $!"
)

proc = subprocess.run(
    ["bash", "-lc", bash_cmd],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

pid = proc.stdout.strip().splitlines()[-1]
PID_PATH.write_text(pid)

print("\nLaunched DFD-HR test PID:", pid)
print("PID saved to:", PID_PATH)

In [5]:
from pathlib import Path
import subprocess
import pandas as pd
import time

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")
OUT_DIR = PROJECT_ROOT / "outputs_dfdhr_wilddeepfake_latest_test"
LOG_PATH = OUT_DIR / "dfdhr_latest_test.log"
PID_PATH = OUT_DIR / "dfdhr_latest_test.pid"

if PID_PATH.exists():
    pid = PID_PATH.read_text().strip()
    print("PID:", pid)
    print(subprocess.run(
        ["bash", "-lc", f"ps -p {pid} -o pid,ppid,etime,stat,%cpu,%mem,rss,cmd || true"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    ).stdout)

print("\nGPU:")
print(subprocess.run(
    ["bash", "-lc", "nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv,noheader,nounits || true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
).stdout or "No GPU process found.")

print("\nOutput files:")
for name in [
    "test_metrics.csv",
    "test_frame_predictions.csv",
    "test_video_predictions.csv",
    "test_threshold_report.csv",
    "test_inference_time.csv",
    "dataset_summary.csv",
    "dfdhr_latest_test.log",
]:
    p = OUT_DIR / name
    if p.exists():
        print(f"FOUND   {name:35s} size_mb={p.stat().st_size/1024**2:.3f} age_min={(time.time()-p.stat().st_mtime)/60:.1f}")
    else:
        print(f"MISSING {name}")

print("\nMetrics:")
for name in ["test_metrics.csv", "metrics.csv"]:
    p = OUT_DIR / name
    if p.exists():
        display(pd.read_csv(p))
        break

print("\nLog tail:")
if LOG_PATH.exists():
    lines = LOG_PATH.read_text(errors="replace").splitlines()
    print("Log lines:", len(lines))
    for line in lines[-120:]:
        print(line)
else:
    print("No log yet.")

PID: 1939
    PID    PPID     ELAPSED STAT %CPU %MEM   RSS CMD


GPU:
No GPU process found.

Output files:
MISSING test_metrics.csv
MISSING test_frame_predictions.csv
MISSING test_video_predictions.csv
MISSING test_threshold_report.csv
MISSING test_inference_time.csv
FOUND   dataset_summary.csv                 size_mb=4.403 age_min=369.1
FOUND   dfdhr_latest_test.log               size_mb=0.006 age_min=369.1

Metrics:

Log tail:
Log lines: 94
Device: cuda
Metadata: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv
Checkpoint: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wilddeepfake_train_no_amp_cc/latest_checkpoint.pt
Output: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wilddeepfake_latest_test
Dropped missing paths: 0

Sampling videos: 100%|██████████| 806/806 [00:07<00:00, 108.39it/s]
Rows: 25792
Videos/groups: 806
col_0  count
label       
0      12672
1      13120
Traceback (most recent 